# 实验 1　文本导入与语料认识

## 任务要求

1. 读取《三国演义》纯文本，检查编码是否正确，打印前 200—300 个字符人工确认。
2. 统计全书总字符数、总行数、非空行数，区分“行”和“段”的定义。
3. 记录观察到的格式问题：空行、广告、注释、特殊符号或回目标题排版特点。
4. 用 3—5 句话说明：这份文本是否已经适合直接做 NLP 分析，为什么。

## 数据与工具

- 原始语料：`data/raw/sanguoyanyi.txt`
- 共享模块：`src/sanguo_nlp/`（路径、语料读取与清洗）
- 输出目录：`outputs/tables/`、`outputs/figures/`

In [1]:
# 让 notebook 在 notebooks/ 或仓库根目录下都能找到 src/ 包
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "sanguo_nlp").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from sanguo_nlp.paths import FIGURES_DIR, TABLES_DIR, ensure_output_dirs

ensure_output_dirs()
print("已定位仓库根目录:", ROOT.name)

已定位仓库根目录: sanguo-nlp-lab


## 1. 读取语料并人工确认

先看数据再写算法：确认编码、观察开头的卷名、作者、卷首词与第一回标题。

In [2]:
from sanguo_nlp.corpus import read_raw

text = read_raw()
print(text[:300])

﻿三国演义 上卷
罗贯中

滚滚长江东逝水，浪花淘尽英雄。是非成败转头空。青山依旧在，几度夕阳红。
白发渔樵江渚上，惯看秋月春风。一壶浊酒喜相逢。古今多少事，都付笑谈中。
--调寄《临江仙》
第一回  宴桃园豪杰三结义  斩黄巾英雄首立功
话说天下大势，分久必合，合久必分。周末七国分争，并入于秦。及秦灭之后，楚、汉分争，又并入于汉。汉朝自高祖斩白蛇而起义，一统天下，后来光武中兴，传至献帝，遂分为三国。推其致乱之由，殆始于桓、灵二帝。桓帝禁锢善类，崇信宦官。及桓帝崩，灵帝即位，大将军窦武、太傅陈蕃共相辅佐。时有宦官曹节等弄权，窦武、陈蕃谋诛之，机事不密，反为所害，中涓自此愈横。
建宁二年四月望日


## 2. 基础统计

- **字符**：文本本身的最小单位，适合衡量语料规模。
- **行**：由换行符切分，适合检查格式与排版。
- **段**：由空行分隔的语义块，适合观察叙事结构。

In [3]:
import re

lines = text.splitlines()
paragraphs = [p for p in re.split(r"\n\s*\n", text) if p.strip()]
headings = re.findall(r"^[ \t]*第[一二三四五六七八九十百千零0-9]+回", text, re.MULTILINE)

stats = {
    "总字符数": len(text),
    "去掉空白后的字符数": len(re.sub(r"\s", "", text)),
    "总行数": len(lines),
    "非空行数": sum(1 for line in lines if line.strip()),
    "自然段数（按空行切分）": len(paragraphs),
    "回目标题数": len(headings),
}
for key, value in stats.items():
    print(f"{key}: {value}")

总字符数: 602415
去掉空白后的字符数: 600348
总行数: 1587
非空行数: 1585
自然段数（按空行切分）: 3
回目标题数: 120


In [4]:
import pandas as pd

stats_df = pd.DataFrame(stats.items(), columns=["指标", "数值"])
stats_df.to_csv(TABLES_DIR / "corpus_stats.csv", index=False, encoding="utf-8-sig")
stats_df

,指标,数值
0,总字符数,602415
1,去掉空白后的字符数,600348
2,总行数,1587
3,非空行数,1585
4,自然段数（按空行切分）,3
5,回目标题数,120


## 3. 格式问题观察

重点检查 BOM、全角空格、行尾空白、正文中混入的目录式标记，以及回目标题的排版是否统一。

In [5]:
print("是否含 UTF-8 BOM:", text[:1] == "\ufeff")
print("全角空格数量:", text.count("\u3000"))
print("行尾带多余空格的行数:", sum(1 for line in lines if line != line.rstrip()))
print("空行数量:", len(lines) - stats["非空行数"])
print("普通空格数量:", text.count(" "))
print("正文中的 '#' 残留标记:", re.findall(r"#[^\n]{0,20}", text))

print("\n回目标题排版示例（前 3 条与后 2 条）:")
for heading in headings[:3] + headings[-2:]:
    print("  ", heading)

是否含 UTF-8 BOM: True
全角空格数量: 2
行尾带多余空格的行数: 0
空行数量: 2
普通空格数量: 478
正文中的 '#' 残留标记: ['#第四十一回']

回目标题排版示例（前 3 条与后 2 条）:
   第一回
   第二回
   第三回
   第一百十九回
   第一百二十回


## 4. 结论与实验记录

**结论**

- 文本是 UTF-8 编码但带 BOM，开头有“三国演义 上卷 / 罗贯中 / 卷首词”等正文以外的内容；第 40 回结尾还混入了一处 `#第四十一回` 的目录式残留。
- 全书共 120 个回目标题。文本几乎不使用空行分段（全篇仅有 3 处空行），因此用空行切出的“自然段”只有 3 个；在本语料中**一个非空行更接近一个段落**，行与段基本重合。
- 语料基本没有乱码，但存在上述少量噪声，**不能直接做 NLP 分析**，需要先经过实验 2 的清洗与按回切分。

**实验记录**

- 遇到的问题：在不同 Python 解释器与 Jupyter kernel 之间切换时出现过模块找不到、`SSL: CERTIFICATE_VERIFY_FAILED` 等环境问题；解决办法是固定使用一个独立的 Python 3.10 环境，并统一用该环境注册 kernel。
- AI 工具使用记录：用 AI 助手解读报错信息、生成 `len()` / `splitlines()` 等基础统计代码；所有结论均在本地运行核对，并人工确认前 300 个字符。